# BloomCast — model training (Colab or Kaggle, GPU)

Trains the production model on **real Tick Tick Bloom in-situ labels** and exports serving artifacts to `/kaggle/working/bloomcast_artifacts/`.

**Before running (both platforms):**
1. Attach the dataset (competition CSVs or CAML `.sb`) — Colab: upload to Drive `bloomcast/data/` or session `/content/`; Kaggle: Add input dataset.
2. Colab: mount Drive when asked (run cells top to bottom) and add the `GITHUB_TOKEN` secret only if the repo is private (key icon → Secrets).
3. Kaggle: internet ON, GPU accelerator, `GITHUB_TOKEN` secret if private.
4. Keep this tab open until the end (Colab free has no background runs) — if disconnected, just re-run: cache + Optuna resume from the work dir.

**Before running (old list — superseded above):**
1. Attach the competition CSVs (`train_labels.csv`, `metadata.csv` — download once from drivendata.org/competitions/143) as a Kaggle input dataset, **or** set `TICKTICKBLOOM_DIR` to their path.
2. Turn **internet ON** (the weather join pulls Open-Meteo archive data; cached to `/kaggle/working` so reruns are free).
3. Accelerator **GPU** recommended (torch CNN trains on it); CPU works, just slower.

Pipeline: labels → weather join → Optuna-tuned LightGBM → quantile models → GPU CNN → ensemble → calibration → export + verify.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# Training-only deps (serving never needs these)
!pip install -q lightgbm scikit-learn pandas httpx scipy optuna shap
import warnings as _w
_w.filterwarnings("ignore", message=".*valid feature names.*")
_w.filterwarnings("ignore", category=UserWarning, module="sklearn")

# --- platform first: everything else hangs off these two ---
# Fail-fast git: an auth problem must error in seconds, never hang on an
# interactive login prompt.
os.environ["GIT_TERMINAL_PROMPT"] = "0"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            print("Mount Google Drive in the popup, then continue...")
            drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount failed:", e)
if IN_COLAB and Path("/content/drive/MyDrive").is_dir():
    DROOT = Path("/content/drive/MyDrive/bloomcast")
    CODE_ROOTS = [DROOT, Path("/content")]
elif Path("/kaggle/working").is_dir():
    DROOT = Path("/kaggle/working")
    CODE_ROOTS = [Path("/kaggle/input")]
else:
    DROOT = Path(".")
    CODE_ROOTS = [Path(".")]
DROOT.mkdir(parents=True, exist_ok=True)
print("environment:", "colab" if IN_COLAB else "kaggle", "| work dir:", DROOT)

# --- training library: clone the repo, or reuse attached code ---
# Clone destination lives under the platform work dir (never hardcoded).
# Public repos clone anonymously; private needs GITHUB_TOKEN (env, Colab
# Secrets key-icon, or Kaggle Secrets - tried in that order).
REPO = "https://github.com/N-Garai/BloomCast.git"
DEST = DROOT / "BloomCast"
LIB = None
if (DEST / "src" / "backend").is_dir():
    # A stale clone is worse than no clone: old training code + new notebook
    # cells fail with phantom TypeErrors and old throttling behavior.
    import subprocess as _sp
    try:
        _pull = _sp.run(f"git -C {DEST} pull --ff-only", shell=True,
                        capture_output=True, text=True, timeout=120)
        print(((_pull.stdout or "") + (_pull.stderr or "")).strip().splitlines()[-1:])
    except Exception as e:
        print("git pull skipped:", e)
    LIB = str(DEST / "src" / "backend")
    print("library: existing clone (refreshed)")
else:
    token = os.environ.get("GITHUB_TOKEN", "")
    if not token:
        try:
            from google.colab import userdata  # Colab Secrets (key icon)
            token = userdata.get("GITHUB_TOKEN") or ""
        except Exception:
            pass
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient  # Kaggle Secrets
            token = UserSecretsClient().get_secret("GITHUB_TOKEN") or ""
        except Exception:
            pass
    url = REPO if not token else REPO.replace("https://", f"https://{token}@")
    import subprocess as _sp
    _clone = _sp.run(f"git clone {url} {DEST}",
                     shell=True, capture_output=True, text=True)
    print((_clone.stderr or "").strip().splitlines()[-3:])
    if _clone.returncode == 0 and (DEST / "src" / "backend").is_dir():
        LIB = str(DEST / "src" / "backend")
        print("library: cloned")
    elif "403" in (_clone.stderr or ""):
        print("HTTP 403: the token authenticated but may not access this repo. "
              "Check: 1) classic PAT has `repo` scope (not just `public_repo`), "
              "2) fine-grained PAT includes THIS repo with Contents: Read, "
              "3) no trailing space/newline pasted into the secret, "
              "4) token not expired or revoked.")
if LIB is None:
    # Fallback: backend code attached as a dataset (no git needed).
    for _root in CODE_ROOTS:
        if not _root.is_dir():
            continue
        hits = sorted(_root.rglob("ml/training/artifacts.py"))
        for hit in hits:
            backend = hit.parent.parent.parent
            if (backend / "ml").is_dir() and (backend / "features").is_dir():
                LIB = str(backend)
                print("library: attached code at", LIB)
                break
        if LIB:
            break
assert LIB is not None, (
    "no training library found. Either make the repo public (or set a "
    "GITHUB_TOKEN secret: classic PAT, `repo` scope, revoke after use), "
    "or attach the backend code as a dataset."
)
sys.path.insert(0, LIB)
print("library ready")

In [ ]:
from pathlib import Path

# Attached datasets land in platform-specific subdirectories, possibly with
# their own internal folders - so search recursively. Accepts competition
# CSVs OR a CAML SeaBASS .sb file, whichever is found first.
def _has_csvs(p):
    return (p / 'train_labels.csv').exists() and (p / 'metadata.csv').exists()

def _has_sb(p):
    try:
        return any(f.is_file() and f.suffix.lower() == '.sb' for f in p.iterdir())
    except OSError:
        return False

# 0) exact file override (env) - no searching needed when you know the path
_exact = os.environ.get('CAML_EXACT_FILE', '')
DATA_DIR, KIND = None, None
if _exact and Path(_exact).is_file():
    DATA_DIR, KIND = Path(_exact).parent, 'seabass'
    print('exact file:', _exact)
else:
    # 1) diagnose the mount - spellings differ by account/version
    for _cand in ['/content/drive/MyDrive', '/content/drive/My Drive', '/content/drive']:
        print(('dir ' if Path(_cand).is_dir() else 'MISS'), '-', _cand)
    roots = [os.environ.get('TICKTICKBLOOM_DIR', ''), os.environ.get('CAML_DIR', ''),
             str(DROOT / 'data'),
             '/content/drive/MyDrive/BloomCast/data', '/content/drive/My Drive/BloomCast/data',
             '/kaggle/input/datasets/ngarai/tick-tick-bloom', '/kaggle/input',
             '/content']
    for root in roots:
        if not root or not Path(root).is_dir():
            print('skip (not a dir):', root)
            continue
        cands = [Path(root)] + sorted(x for x in Path(root).iterdir() if x.is_dir())
        for cand in cands:
            if _has_csvs(cand):
                DATA_DIR, KIND = cand, 'competition'
                break
            if _has_sb(cand):
                DATA_DIR, KIND = cand, 'seabass'
                break
        if DATA_DIR:
            break
    if DATA_DIR is None:
        print('--- contents of likely parents ---')
        for cand in ['/content/drive/MyDrive/BloomCast', '/content/drive/My Drive/BloomCast', '/content']:
            p = Path(cand)
            print(cand, '->', sorted(x.name for x in p.iterdir()) if p.is_dir() else 'NOT A DIR')
assert DATA_DIR is not None, 'no dataset found - see diagnostics above'
os.environ['TICKTICKBLOOM_DIR'] = str(DATA_DIR)
print('dataset:', DATA_DIR, '| format:', KIND)

# Optional quality filter (CAML manual Table 4): drop samples taken far from
# water - likely mislocated. Unset keeps everything.
_max_d = os.environ.get('CAML_MAX_DISTANCE_M', '')
MAX_DISTANCE_M = float(_max_d) if _max_d.strip() else None

from ingestion.drivendata_loader import load_training_frame
rows = load_training_frame(DATA_DIR, max_distance_m=MAX_DISTANCE_M)

# Deterministic time-box lever: stratified subsample by (label, region),
# date order restored after. Full data is the default; N_SAMPLES=8000 cuts
# the weather join to ~1/3 with the same class/region mix. Reproducible (seed).
_n = os.environ.get('N_SAMPLES', '').strip()
if _n and len(rows) > int(_n):
    import random as _rnd
    _groups = {}
    for _r in rows:
        _groups.setdefault((_r['y'], _r['region']), []).append(_r)
    _quota = int(_n) / len(rows)
    _shuf = _rnd.Random(7)
    _sub = []
    for _g in _groups.values():
        _gg = list(_g)
        _shuf.shuffle(_gg)  # groups arrive date-sorted; shuffle before slicing
        _sub.extend(_gg[:max(1, int(len(_gg) * _quota))])
    rows = sorted(_sub, key=lambda r: r['date'])[:int(_n)]
    print(f'subsampled to {len(rows)} rows (N_SAMPLES={_n})')
n_pos = sum(r['y'] for r in rows)
print('samples:', len(rows), '| positives:', n_pos, f'({n_pos/len(rows):.1%})',
      '| span:', rows[0]['date'], '->', rows[-1]['date'])

In [ ]:
from ml.training.real_labels import build_weather_frame

# Weather is cached beside the output so reruns never re-hit the API
CACHE = DROOT / "weather_cache.csv"
frame = build_weather_frame(rows, cache_path=CACHE)
X, S, y, doy, wb_ids = frame["X"], frame["S"], frame["y"], frame["doy"], frame["wb_ids"]
regions, severity = frame["regions"], frame["severity"]
sw = 0.5 + severity.astype(float) / 5.0  # severe blooms count more (see model card)
print(f"X{X.shape} S{S.shape} positives={y.mean():.1%} skipped={frame['skipped']} regions={regions}")


In [ ]:
from ml.training.tuning import tune_lightgbm

N_TRIALS = int(os.environ.get("N_TRIALS", "30"))
# Persisted trials: an interrupted run resumes instead of restarting.
_opt_db = DROOT / "optuna.db"
_storage = f"sqlite:///{_opt_db}"
import optuna as _opt
_done = 0
if _storage:
    try:
        _done = len(_opt.load_study(study_name='bloomcast-lgbm', storage=_storage).trials)
    except Exception:
        _done = 0
_remain = max(0, N_TRIALS - _done)
print(f'optuna trials: {_done} done, {_remain} remaining')
tuned = tune_lightgbm(X, y, n_trials=max(_remain, 1), n_splits=3, storage=_storage)
if _remain == 0:
    print('study already complete - reusing best params')
print("best Brier:", round(tuned["best_brier"], 4))
print(tuned["params"])

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from ml.training.tuning import train_quantiles
from ml.training.lightgbm_branch import calibrate, ClimatologyBaseline, PersistenceBaseline, WeatherOnlyBaseline
from ml.training.scorecard import generate_scorecard
from ml.inference.predict import _proba, empirical_ci_half, per_region_metrics

P = tuned["params"]
tscv = TimeSeriesSplit(n_splits=5)
oof = np.zeros(len(y))
for tr, te in tscv.split(X):
    m = lgb.LGBMClassifier(**P)
    m.fit(X[tr].astype(np.float64), y[tr], sample_weight=sw[tr])
    oof[te] = m.predict_proba(X[te].astype(np.float64))[:, 1]
oof = np.where(np.isfinite(oof), oof, np.nanmean(oof))
final = lgb.LGBMClassifier(**P)
final.fit(X.astype(np.float64), y, sample_weight=sw)
calibrator = calibrate(oof, y)
ci_half = empirical_ci_half(y, oof)
print(f"OOF AUC/Brier on held-out folds - full scorecard below")

qmodels = train_quantiles(X, y, P, sample_weight=sw)
print("quantiles trained: q05/q95")

from ml.training.lightgbm_branch import LightGBMBranch
branch = LightGBMBranch.__new__(LightGBMBranch)
branch.model = final

clim = ClimatologyBaseline().fit(X, y, doy, wb_ids)
persist = PersistenceBaseline().fit(X, y)
weather = WeatherOnlyBaseline().fit(X, y)
per_region = per_region_metrics(y, oof, wb_ids, regions)
sc = generate_scorecard(y, oof, clim.predict(X, doy, wb_ids), persist.predict(X), weather.predict(X),
                        model_version="v2.2.0-kaggle-labels", sample_size=len(y), horizon_days=5,
                        per_region=per_region)
print(f"OOF AUC={sc['auc']:.4f} Brier={sc['brier']:.4f} hit={sc['hit_rate']:.2%} fa={sc['false_alarm_rate']:.2%} ci_half={ci_half:.3f}")
print("per-region:", {k: (v["auc"], v["n"]) for k, v in per_region.items()})

In [ ]:
from ml.training.torch_cnn import train_torch_cnn, export_to_numpy
from ml.training.cnn_branch import BloomCNN
from ml.training.ensemble import Ensemble

CNN_EPOCHS = int(os.environ.get("CNN_EPOCHS", "15"))
tmodel, hist = train_torch_cnn(S, y, epochs=CNN_EPOCHS, lr=1e-3, batch_size=512)
print(f"CNN on {hist['device']} — loss {hist['loss'][0]:.4f} → {hist['loss'][-1]:.4f}")

np_weights = export_to_numpy(tmodel)
cnn = BloomCNN()
for k, v in np_weights.items():
    setattr(cnn, k, v)
# serving-exactness check: torch vs numpy embeddings must match
_dev = next(tmodel.net.parameters()).device
Xt = __import__("torch").as_tensor(S[:64]).to(_dev)
_, emb_t = tmodel.forward(Xt)
assert np.allclose(emb_t.detach().cpu().numpy(), cnn.predict_embedding(S[:64]), atol=1e-5)
print("torch→numpy export verified exact")

emb_train = cnn.predict_embedding(S)
ens = Ensemble().fit(oof, emb_train, X, y, sample_weight=sw)
print("ensemble fitted")

In [ ]:
import json
from datetime import datetime, timezone
from ml.training.artifacts import save_artifacts
from features.feature_store import FEATURE_NAMES
from ml.inference.predict import _profile_features, _shap_top, _band

OUT = DROOT / "bloomcast_artifacts"
base = _profile_features()[3]
raw = float(final.predict_proba(base.reshape(1, -1).astype(np.float64))[0, 1])
p_head = float(calibrator.transform([raw])[0])
lo_head, hi_head = _band(p_head, base, qmodels, ci_half)
meta = {
    "model_version": "v2.2.0-kaggle-labels",
    "training_source": "tick-tick-bloom",
    "training_note": f"Kaggle-trained on {len(y)} Tick Tick Bloom samples; Optuna-tuned LightGBM + GPU CNN + quantiles.",
    "exported_at": datetime.now(timezone.utc).isoformat(),
    "ci_half": ci_half,
    "oof_auc": sc["auc"],
    "oof_brier": sc["brier"],
    "optuna_params": P,
    "scorecard": json.loads(json.dumps(sc, default=float)),
    "headline": {
        "p_bloom": p_head,
        "ci_lo": lo_head,
        "ci_hi": hi_head,
        "shap_top_features": _shap_top(branch, base),
        "baseline_climatology": float(clim.predict(base.reshape(1, -1), doy[:1], wb_ids[:1])[0]),
    },
}
save_artifacts(OUT, lgbm_branch=branch, calibrator=calibrator, ensemble=ens,
               cnn=cnn, feature_names=FEATURE_NAMES, meta=meta,
               quantiles=qmodels)
print("exported:", sorted(p.name for p in OUT.iterdir()))

# Reload verification — the backend will do exactly this
from ml.training.artifacts import load_artifacts
re = load_artifacts(OUT, FEATURE_NAMES)
assert re is not None and re["meta"]["training_source"] == "tick-tick-bloom"
assert re["meta"].get("has_quantiles") is True
print(f"reload OK — download {OUT} (7 files) and commit it to src/backend/ml/artifacts/")

## After the run

1. Download **`bloomcast_artifacts/`** from `/kaggle/working` (Kaggle → Output → download).
2. Copy the 7 files into **`src/backend/ml/artifacts/`** in the repo and commit.
3. CI enforces the honesty rule (`test_committed_artifacts_are_real`) — only `tick-tick-bloom` provenance is accepted.
4. The next nightly seed + every deployment loads the model and runs **inference only**. No retraining, no labels, no weather join.